# `04_r4s3_usd_behavioral.ipynb` — R4-S3-USD **behavioral subscription-inelasticity test** (framework-relevant arm)

**Spec citations:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.10 — §2.2.B (extended PARTIAL-* labels SCOPED to demonstration-grade, NOT applicable to this verdict), §0.1 CORRECTIONS-S (test inversion), §0.1 CORRECTIONS-I (HAC bandwidth `floor(T^(1/3))`), §0.1 CORRECTIONS-J + CORRECTIONS-U (residual-SD power floor 0.50 at MDES=0.40), §0.7 CORRECTIONS-Y-9 (UTC parity closure), §0.8 CORRECTIONS-Z2 (Task 13 reclassification), §2.3.1 (demonstration-grade pin — N_MIN=38, POWER_MIN=0.50 — Amendment #5), **§0.10 CORRECTIONS-W3 Amendment #8 (Z3 REVERTED to LAGGED canonical recipe)**.

**Plan:** `docs/plans/2026-05-16-ai-cost-factor-model-plan.md` Task 14 + Task 14-bis.

## CORRECTIONS-S preamble (framework-relevant arm)

Per **CORRECTIONS-S**, this notebook is the **framework-relevant arm** of R4-S3. The pre-registered direction is *"null cannot be rejected"* (subscription-regime inelasticity); **rejecting the null** — in either direction — reveals a behavioral channel despite zero marginal cost, which is itself an informative finding. **Both outcomes feed M-design assumptions; neither routes to FAIL.**

**Inversion vs. R4-S3-COP (Task 13).** R4-S3-COP was the *consistency check* (mechanical FX pass-through under USD-denominated cost data → COP). Its v0.2.7 verdict was CONSISTENCY-FAIL, **reclassified to REGIME-CONDITIONAL** per v0.2.8 §0.8 Z2: both preconditions hold — (A) additive-identity check passes at 1.78e-15 absolute error (panel construction is exact `ln COP = ln USD + ln TRM`), and (B) FX β shrinks toward zero in the deflated regression. Per v0.2.10 §0.10 Amendment #6/#7, R4-S3-COP's REGIME-CONDITIONAL FAIL is UNINFORMATIVE and is REMOVED from the R5 corroboration list. Task 14 is unblocked; today's test isolates the **behavioral** USD-side response.

**Sample-floor caveat (demonstration-grade — Amendment #5).** Current panel has **N=28 weekdays** post-first-diff. §2.3.1 (demonstration-grade) pins N_MIN=38, POWER_MIN=0.50. The PARTIAL-* labels in §2.2.B are SCOPED to demonstration-grade and were premised on the power gate passing under the contemporaneous recipe; under the v0.2.10 reverted lagged recipe the power gate fires FIRST and routes to POWER-HALT (PARTIAL-* labels NOT applicable). No silent override of any gate.

**π̂ disclaimer (CORRECTIONS-Y-6).** All notional-cost figures use a fixed `ephemeral_pi_share = 0.398977` (audited point estimate from the snapshot panel); the actual ephemeral cache-write proportion at runtime is unobservable in ccusage logs and is therefore frozen as a calibrated constant, not a measured time-series.

**Y-9 closure (v0.2.7).** Cost data has been verified at **1.000000× ccusage parity** post-UTC alignment (timezone-artifact resolved); LHS and RHS series are apples-to-apples on the same UTC weekday cadence.

**v0.2.10 §0.10 Amendment #8 — Z3 REVERTED to LAGGED canonical recipe (NEW).** Task 14's v0.2.8 implementation computed power using *lagged-tokens* partialling and reported `power = 0.1745`, which routed the verdict to POWER-HALT. v0.2.9 §0.9 had reversed this to pin the *contemporaneous* recipe (power 0.71) as canonical on chronological-precedence + first-principles grounds. **v0.2.10 §0.10 Amendment #8 REVERTS that pin**: the canonical recipe is now **LAGGED tokens partialling** — matching R4-S3-USD's actual k=1 spec lag structure (the regression's own residuals). The v0.2.9 "first-principles" defense overstated the case; honest-disclosure-as-process applies. This is a **TIGHTENING** of the verdict's confidence claim (the harder verdict passes), not a loosening; no threshold is tuned. Trio 4 computes BOTH recipes for transparency; the **LAGGED recipe gates the verdict**. Anticipated outcome: power 0.17 < 0.50 → **POWER-HALT** (anticipated per CORRECTIONS-U).


## Cell 2 — Anti-fishing pins (declared PRE-DATA)

Constants below are pinned **before** any data read. They must not be tuned post-hoc; doing so is silent-fishing per `feedback_pathological_halt_anti_fishing_checkpoint.md`.

In [1]:
from __future__ import annotations

import numpy as np

# ANTI-FISHING PINS — set PRE-DATA, DO NOT TUNE
LAG_PRIMARY = 1                    # §2.2.B primary lag (k=1)
LAG_SENSITIVITY = 5                # §2.2.B sensitivity lag (k=5)
ALPHA_LEVEL = 0.05                 # TWO-SIDED test threshold (CORRECTIONS-S)
TOKENS_PROXY = 'output_tok'        # consistent with Task 13
USDCOP_SERIES = 'trm_cop_per_usd'  # Banrep TRM
COST_SERIES = 'notional_cost_usd'  # USD-denominated; the behavioral LHS

# Sample-floor gate (§2.2.B verdict logic)
N_MIN = 38

# Power gate (CORRECTIONS-J + CORRECTIONS-U — residual SD, MDES=0.40)
POWER_THRESHOLD = 0.50
MDES_RESID_SD = 0.40
POWER_MC_DRAWS = 2000
POWER_SEED = 20260517            # reproducibility

# HAC bandwidth pinned PRE-DATA as floor(T^(1/3)) per CORRECTIONS-I (Andrews 1991)
def hac_bandwidth(t_lag: int) -> int:
    """Andrews (1991) rule-of-thumb HAC bandwidth: floor(T^(1/3))."""
    return int(np.floor(t_lag ** (1 / 3)))

# Verdict label set (closed; no other labels permitted post-hoc)
VERDICT_LABELS = {
    'REJECT_NULL',
    'FAIL_TO_REJECT',
    'PARTIAL-REJECT',         # p<0.05 but N<38; disclosed
    'PARTIAL-FAIL_TO_REJECT', # p>=0.05 but N<38; disclosed
    'POWER-HALT',
    'HALT',
}

print('=' * 60)
print('ANTI-FISHING PINS (pre-data)')
print('=' * 60)
print(f'LAG_PRIMARY        = {LAG_PRIMARY}')
print(f'LAG_SENSITIVITY    = {LAG_SENSITIVITY}')
print(f'ALPHA_LEVEL        = {ALPHA_LEVEL} (TWO-SIDED per CORRECTIONS-S)')
print(f'TOKENS_PROXY       = {TOKENS_PROXY}')
print(f'USDCOP_SERIES      = {USDCOP_SERIES}')
print(f'COST_SERIES        = {COST_SERIES}')
print(f'N_MIN              = {N_MIN}  (§2.2.B verdict-eligibility floor)')
print(f'POWER_THRESHOLD    = {POWER_THRESHOLD}  at MDES={MDES_RESID_SD} residual-SD')
print(f'POWER_MC_DRAWS     = {POWER_MC_DRAWS}')
print(f'POWER_SEED         = {POWER_SEED}')
print(f'HAC bandwidth      = floor(T^(1/3))  per CORRECTIONS-I')
print(f'VERDICT_LABELS     = {sorted(VERDICT_LABELS)}')

ANTI-FISHING PINS (pre-data)
LAG_PRIMARY        = 1
LAG_SENSITIVITY    = 5
ALPHA_LEVEL        = 0.05 (TWO-SIDED per CORRECTIONS-S)
TOKENS_PROXY       = output_tok
USDCOP_SERIES      = trm_cop_per_usd
COST_SERIES        = notional_cost_usd
N_MIN              = 38  (§2.2.B verdict-eligibility floor)
POWER_THRESHOLD    = 0.5  at MDES=0.4 residual-SD
POWER_MC_DRAWS     = 2000
POWER_SEED         = 20260517
HAC bandwidth      = floor(T^(1/3))  per CORRECTIONS-I
VERDICT_LABELS     = ['FAIL_TO_REJECT', 'HALT', 'PARTIAL-FAIL_TO_REJECT', 'PARTIAL-REJECT', 'POWER-HALT', 'REJECT_NULL']


## Decision-citation block — two-sided test choice

- **Reference:** spec §2.2.B + §0.1 CORRECTIONS-S.
- **Why:** the subscription-regime null is "no behavioral response of USD-side cost vol to FX vol." Under a fixed-fee invoicing regime, marginal cost in USD is zero, so $\alpha_1^{USD}$ should not be meaningfully different from zero — *in either direction*. Rejecting EITHER direction is the behavioral-channel finding (the wage-earner-as-user might shift token consumption defensively when COP weakens, even though their USD bill is unchanged). A one-sided test would silently throw away half the inferential surface.
- **Relevance:** anti-fishing pre-pinning. Switching to one-sided after seeing the sign of $\hat\alpha_1^{USD}$ is silent-fishing and is explicitly banned.
- **Connection:** feeds the verdict cell; `p_two_sided` (from `statsmodels`) is the inputs to the REJECT_NULL / FAIL_TO_REJECT gate.

## Trio 1 — Construct $|\Delta\ln\text{NotionalCost}^{USD}|$, $|\Delta\ln\text{USDCOP}|$, $|\Delta\ln\text{Tokens}|$

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B equation (lines 1351–1353).
- **Why:** the spec equation is symmetric to §2.2.A in structure but with `notional_cost_usd` on the LHS. The absolute log first-difference of three series is the canonical "volatility-on-volatility" RHS for FX-cost transmission (Pair-D precedent + CORRECTIONS-S).
- **Relevance:** same series-construction recipe as Task 13 — but LHS is now USD-denominated, isolating the *behavioral* (token-consumption) channel from the *mechanical* (FX-translation) channel that dominates the COP arm.
- **Connection:** outputs three numpy arrays of length $T = N - 1$ feeding the regression trios.

In [2]:
from pathlib import Path

import polars as pl

PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')
df = pl.read_parquet(PANEL_PATH)
print(f'panel shape: {df.shape}')
print(f'date window: {df["date_utc"].min()} -> {df["date_utc"].max()}')
print(f'columns: {df.columns}')

# Compute absolute first-diff of natural log for the three series
abs_dln_usd = df[COST_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_trm = df[USDCOP_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_tok = df[TOKENS_PROXY].log().diff().abs().drop_nulls().to_numpy()

T = len(abs_dln_usd)
assert len(abs_dln_trm) == T == len(abs_dln_tok), 'series-length mismatch'

print()
print(f'T (post first-diff)                  = {T}')
print(f'|Δln NotionalCost^USD|  range        = [{abs_dln_usd.min():.6f}, {abs_dln_usd.max():.6f}]')
print(f'|Δln USDCOP|             range        = [{abs_dln_trm.min():.6f}, {abs_dln_trm.max():.6f}]')
print(f'|Δln Tokens (output_tok)| range       = [{abs_dln_tok.min():.6f}, {abs_dln_tok.max():.6f}]')
print(f'\nSpec floor N_MIN={N_MIN}; observed T={T} → {"PASS" if T >= N_MIN else "BELOW FLOOR (PARTIAL-* verdict path)"}')

panel shape: (29, 11)
date window: 2026-01-06 -> 2026-05-14
columns: ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd', 'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h', 'cache_read', 'n_messages', 'ephemeral_pi_share']

T (post first-diff)                  = 28
|Δln NotionalCost^USD|  range        = [0.000832, 5.483817]
|Δln USDCOP|             range        = [0.000632, 0.023392]
|Δln Tokens (output_tok)| range       = [0.000000, 7.697189]

Spec floor N_MIN=38; observed T=28 → BELOW FLOOR (PARTIAL-* verdict path)


### Interpretation — Trio 1

Three numpy arrays of length $T$ constructed from the production panel. $T$ is reported above; if $T < 38$ the verdict cell will route to a PARTIAL-* label. The USD-side cost vol range is large (driven by sparse low-cost weekdays mixing with high-load weekdays), consistent with subscription invoice variance dominated by message-volume rather than FX.

## Trio 2 — Primary regression k=1 with HAC (TWO-SIDED test)

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B + CORRECTIONS-S (two-sided) + CORRECTIONS-I (HAC `floor(T^(1/3))`).
- **Why:** regress $|\Delta\ln\text{NotionalCost}^{USD}|_t$ on $|\Delta\ln\text{USDCOP}|_{t-1}$ and $|\Delta\ln\text{Tokens}|_{t-1}$ with Newey-West HAC SE; **two-sided test** on $\alpha_1^{USD} = 0$. The token-vol control is mechanical (more tokens → more invoice variance); the FX-vol coefficient is the behavioral channel of interest.
- **Relevance:** k=1 is the pre-pinned primary lag (one-trading-day FX update assimilation); HAC L = $\lfloor T_{lag}^{1/3} \rfloor$ controls for residual autocorrelation under small-$T$ daily cadence. The v0.1.0 fixed $L=12$ would over-shrink at $T_{lag} \approx 28$ and is explicitly rejected by CORRECTIONS-I.
- **Connection:** `p_two_sided_primary` feeds the verdict gate; `model_k1.resid` SD feeds Trio 4's power measurement.

In [3]:
import statsmodels.api as sm

# Lag-aligned arrays for k=1 primary
T_lag_primary = T - LAG_PRIMARY
hac_L_primary = hac_bandwidth(T_lag_primary)
print(f'T_lag (k={LAG_PRIMARY}) = {T_lag_primary}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_primary ** (1/3):.4f}) = {hac_L_primary}')

y_primary = abs_dln_usd[LAG_PRIMARY:]              # LHS at time t
x1_primary = abs_dln_trm[:-LAG_PRIMARY]            # |Δln USDCOP|_{t-1}
x2_primary = abs_dln_tok[:-LAG_PRIMARY]            # |Δln Tokens|_{t-1}
assert len(y_primary) == len(x1_primary) == len(x2_primary) == T_lag_primary, 'alignment broken'

X_primary = sm.add_constant(np.column_stack([x1_primary, x2_primary]))
model_k1 = sm.OLS(y_primary, X_primary).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_primary})
print()
print(model_k1.summary())

# Extract pre-pinned coefficient of interest (α₁^USD)
alpha_1_usd_primary = float(model_k1.params[1])
se_alpha_1_primary = float(model_k1.bse[1])
t_stat_primary = alpha_1_usd_primary / se_alpha_1_primary
p_two_sided_primary = float(model_k1.pvalues[1])   # statsmodels reports TWO-SIDED p-values

print()
print(f'α̂₁^USD (k=1)      = {alpha_1_usd_primary:+.6f}')
print(f'HAC SE            = {se_alpha_1_primary:.6f}')
print(f't-stat            = {t_stat_primary:+.4f}')
print(f'p_two_sided       = {p_two_sided_primary:.6f}')
print(f'reject at α=0.05? = {p_two_sided_primary < ALPHA_LEVEL}')

T_lag (k=1) = 27
HAC L  = floor(T_lag^(1/3)) = floor(3.0000) = 3

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.076
Method:                 Least Squares   F-statistic:                    0.2527
Date:                Mon, 18 May 2026   Prob (F-statistic):              0.779
Time:                        08:30:13   Log-Likelihood:                -41.645
No. Observations:                  27   AIC:                             89.29
Df Residuals:                      24   BIC:                             93.18
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation — Trio 2 (k=1 primary)

Estimate, HAC SE, t-stat, and two-sided p-value reported above. Pass/fail at the pre-pinned $\alpha = 0.05$ level is determined by whether `p_two_sided_primary < 0.05`. Under CORRECTIONS-S framing: rejection here = behavioral channel; non-rejection = subscription inelasticity confirmed.

## Trio 3 — Sensitivity regression k=5 with HAC

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B sensitivity lag.
- **Why:** same recipe with $k = 5$ to test whether the behavioral response operates over a trading-week horizon (delayed reaction to sustained FX moves vs. one-day reaction).
- **Relevance:** k=5 is pre-pinned for sensitivity *only*; it does not override the k=1 verdict. Divergence between k=1 and k=5 informs Stage-3 M-design temporal assumptions.
- **Connection:** reported alongside k=1 in the verdict cell.

In [4]:
T_lag_sens = T - LAG_SENSITIVITY
hac_L_sens = hac_bandwidth(T_lag_sens)
print(f'T_lag (k={LAG_SENSITIVITY}) = {T_lag_sens}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_sens ** (1/3):.4f}) = {hac_L_sens}')

y_sens = abs_dln_usd[LAG_SENSITIVITY:]
x1_sens = abs_dln_trm[:-LAG_SENSITIVITY]
x2_sens = abs_dln_tok[:-LAG_SENSITIVITY]
assert len(y_sens) == len(x1_sens) == len(x2_sens) == T_lag_sens, 'alignment broken'

X_sens = sm.add_constant(np.column_stack([x1_sens, x2_sens]))
model_k5 = sm.OLS(y_sens, X_sens).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_sens})
print()
print(model_k5.summary())

alpha_1_usd_sens = float(model_k5.params[1])
se_alpha_1_sens = float(model_k5.bse[1])
t_stat_sens = alpha_1_usd_sens / se_alpha_1_sens
p_two_sided_sens = float(model_k5.pvalues[1])

print()
print(f'α̂₁^USD (k=5)      = {alpha_1_usd_sens:+.6f}')
print(f'HAC SE            = {se_alpha_1_sens:.6f}')
print(f't-stat            = {t_stat_sens:+.4f}')
print(f'p_two_sided       = {p_two_sided_sens:.6f}')

T_lag (k=5) = 23
HAC L  = floor(T_lag^(1/3)) = floor(2.8439) = 2

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.091
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.357
Date:                Mon, 18 May 2026   Prob (F-statistic):              0.280
Time:                        08:30:13   Log-Likelihood:                -35.781
No. Observations:                  23   AIC:                             77.56
Df Residuals:                      20   BIC:                             80.97
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation — Trio 3 (k=5 sensitivity)

Estimate and two-sided p reported above. Convergence with k=1 reinforces the verdict; divergence flags lag-dependent behavioral response (worth noting in disposition memo but does not change the primary verdict).

## Trio 4 — Power measurement on residual SD (CORRECTIONS-J + CORRECTIONS-U + v0.2.10 §0.10 CORRECTIONS-W3 Amendment #8)

### Decision-citation block (4-part)

- **Reference:** spec §2.2.B + CORRECTIONS-J + CORRECTIONS-U + **v0.2.10 §0.10 CORRECTIONS-W3 Amendment #8** (Z3 REVERTED to lagged-tokens canonical recipe; v0.2.9 §0.9 contemporaneous pin withdrawn).
- **Why:** post-control residual variance is the correct yardstick for MDES; using raw-LHS SD double-counts variance the token-vol control already explains. Power threshold pinned at 0.50 (demonstration-grade per §2.3.1 Amendment #5). **CORRECTIONS-Z3 Amendment #8 mandates the LAGGED tokens recipe as canonical**, matching R4-S3-USD's actual k=1 specification's lag structure (the regression's own residuals from $|\\Delta\\ln \\text{NotionalCost}^{USD}|_t$ on $|\\Delta\\ln \\text{USDCOP}|_{t-1}, |\\Delta\\ln \\text{Tokens}|_{t-1}$). The v0.2.9 contemporaneous-tokens defense overstated the case; that recipe is a defensible alternative reading but its v0.2.9 selection over the lagged reading was not first-principles-required — it was the recipe that survived the power gate. v0.2.10 reverts to honest disclosure: use the SAME lagged regressor for the power-calc residual that the regression itself uses.
- **Relevance:** both recipes computed for transparency per Amendment #8 (the "dual-recipe disclosure" of v0.2.9 §0.9 is RETAINED in v0.2.10; what changes is WHICH recipe is canonical). The **LAGGED** power figure gates the verdict; the **CONTEMPORANEOUS** figure is documented for transparency. Pinned MC draws (B=2000) and seed (20260517) prevent post-hoc draw-tuning.
- **Connection:** the **lagged** power figure gates the verdict cell. If canonical (lagged) power < 0.50, verdict routes to **POWER-HALT** (anticipated per CORRECTIONS-U). The disposition memo at `notebooks/dev_ai_cost_v2/dispositions/2026-05-18-task14-power-halt.md` MUST be authored when POWER-HALT fires (CORRECTIONS-U pre-enumerated routing).

In [5]:
# =====================================================================
# v0.2.10 §0.10 CORRECTIONS-W3 Amendment #8 — Z3 REVERTED to LAGGED
# Canonical:   LAGGED tokens partialling          (matches R4-S3-USD k=1 spec lag)
# Alternative: CONTEMPORANEOUS tokens partialling (Task 11 EDA Trio 5 recipe, diagnostic)
# Both recipes computed for transparency per Amendment #8's "dual-recipe disclosure".
# =====================================================================

# ---------------------------------------------------------------------
# Recipe A — CANONICAL (lagged tokens partialling) per v0.2.10 Amendment #8
# Partialling regressor: |Δln tok|_{t-1} on lag-aligned slice (T_lag = 27).
# This matches the actual R4-S3-USD k=1 regression's lag structure — the
# residuals partial out the SAME regressor (lagged) that the regression uses.
# ---------------------------------------------------------------------
X_partial_lagged = sm.add_constant(x2_primary)                # const + |Δln tok|_{t-1}
model_partial_lagged = sm.OLS(y_primary, X_partial_lagged).fit()
residuals_lagged = model_partial_lagged.resid
residual_sd_lagged = float(residuals_lagged.std(ddof=1))
mdes_effect_lagged = MDES_RESID_SD * residual_sd_lagged

print('--- Recipe A: LAGGED tokens partialling (CANONICAL per v0.2.10 Amendment #8) ---')
print(f'Residual SD (lagged partialling)   = {residual_sd_lagged:.6f}')
print(f'MDES effect (0.40 × residual SD)   = {mdes_effect_lagged:.6f}')

rng_lagged = np.random.default_rng(POWER_SEED)
B = POWER_MC_DRAWS
rejected_lagged = 0

x1_obs = x1_primary.copy()
x2_obs = x2_primary.copy()
intercept_obs = float(model_k1.params[0])
alpha2_obs = float(model_k1.params[2])

for b in range(B):
    eps = rng_lagged.normal(loc=0.0, scale=residual_sd_lagged, size=T_lag_primary)
    y_sim = intercept_obs + mdes_effect_lagged * x1_obs + alpha2_obs * x2_obs + eps
    X_sim = sm.add_constant(np.column_stack([x1_obs, x2_obs]))
    try:
        m_sim = sm.OLS(y_sim, X_sim).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_primary})
        if m_sim.pvalues[1] < ALPHA_LEVEL:
            rejected_lagged += 1
    except (np.linalg.LinAlgError, ValueError):
        continue

power_lagged = rejected_lagged / B
print(f'Monte Carlo draws B                = {B}')
print(f'Rejections at α=0.05               = {rejected_lagged}')
print(f'Power (lagged recipe, CANONICAL)   = {power_lagged:.4f}')
print()

# ---------------------------------------------------------------------
# Recipe B — ALTERNATIVE (contemporaneous tokens partialling, DIAGNOSTIC)
# Per v0.2.10 §0.10 Amendment #8, this is RETAINED as a documented diagnostic
# (Task 11 EDA Trio 5 recipe; v0.2.9 §0.9 had pinned it as canonical but that
# pin is WITHDRAWN under Amendment #8). Reported alongside canonical for
# transparency; does NOT gate the verdict.
# Partialling regressor: |Δln tokens|_t on the FULL post-first-diff series (T = 28).
# ---------------------------------------------------------------------
y_full = abs_dln_usd                                          # length T
x_full = abs_dln_tok                                          # length T (contemporaneous)
assert len(y_full) == len(x_full) == T, 'contemporaneous alignment broken'

X_partial_contemp = sm.add_constant(x_full)
model_partial_contemp = sm.OLS(y_full, X_partial_contemp).fit()
residuals_contemp = model_partial_contemp.resid
# Task 11 uses sqrt(SSR / (T - 2)); equivalent to np.std with ddof=2.
residual_sd_contemp = float(np.sqrt((residuals_contemp ** 2).sum() / (len(residuals_contemp) - 2)))
mdes_effect_contemp = MDES_RESID_SD * residual_sd_contemp

print('--- Recipe B: CONTEMPORANEOUS tokens partialling (ALTERNATIVE / DIAGNOSTIC) ---')
print(f'Residual SD (contemporaneous)      = {residual_sd_contemp:.6f}')
print(f'MDES effect (0.40 × residual SD)   = {mdes_effect_contemp:.6f}')

sd_x_contemp = float(np.std(x_full, ddof=1))
beta_true_contemp = MDES_RESID_SD * residual_sd_contemp / sd_x_contemp
hac_L_contemp = hac_bandwidth(T)
print(f'HAC L (contemporaneous, T={T}) = {hac_L_contemp}')
print(f'true β under H1 (contemporaneous)  = {beta_true_contemp:.6f}')

rng_contemp = np.random.default_rng(POWER_SEED)
rejected_contemp = 0
X_contemp_const = sm.add_constant(x_full)

from scipy import stats as sps  # noqa: E402
z_crit = sps.norm.ppf(1 - ALPHA_LEVEL / 2)

for b in range(B):
    eps = rng_contemp.normal(loc=0.0, scale=residual_sd_contemp, size=T)
    y_sim = beta_true_contemp * x_full + eps
    try:
        m_sim = sm.OLS(y_sim, X_contemp_const).fit(
            cov_type='HAC', cov_kwds={'maxlags': hac_L_contemp}
        )
        if abs(m_sim.tvalues[1]) > z_crit:
            rejected_contemp += 1
    except (np.linalg.LinAlgError, ValueError):
        continue

power_contemporaneous = rejected_contemp / B
print(f'Monte Carlo draws B                = {B}')
print(f'Rejections at α=0.05               = {rejected_contemp}')
print(f'Power (contemporaneous, diagnostic) = {power_contemporaneous:.4f}')
print()

# ---------------------------------------------------------------------
# Summary table + canonical alias for downstream verdict cell
# ---------------------------------------------------------------------
print('=' * 64)
print('POWER RECIPE COMPARISON (v0.2.10 §0.10 Amendment #8 — Z3 REVERTED)')
print('=' * 64)
print(f'  Canonical   (LAGGED)        power = {power_lagged:.4f}'
      f'  gate = {"PASS" if power_lagged >= POWER_THRESHOLD else "FAIL"}')
print(f'  Alternative (contemporaneous) power = {power_contemporaneous:.4f}'
      f'  gate = {"PASS" if power_contemporaneous >= POWER_THRESHOLD else "FAIL"}')
print(f'  Power threshold (demonstration-grade) = {POWER_THRESHOLD}')
print('=' * 64)
print('Per v0.2.10 §0.10 Amendment #8 (REVERTS v0.2.9 §0.9):')
print('  CANONICAL = LAGGED tokens partialling (matches regression k=1 lag structure).')
print('  The CONTEMPORANEOUS recipe is preserved as documented diagnostic.')
print('  The LAGGED recipe gates the verdict.')

# v0.2.10 Amendment #8 — measured_power is the canonical (LAGGED) figure
# that feeds the verdict gate. (Previously contemporaneous under v0.2.9 §0.9.)
measured_power = power_lagged
residual_sd = residual_sd_lagged
mdes_effect = mdes_effect_lagged


--- Recipe A: LAGGED tokens partialling (CANONICAL per v0.2.10 Amendment #8) ---
Residual SD (lagged partialling)   = 1.155794
MDES effect (0.40 × residual SD)   = 0.462317


Monte Carlo draws B                = 2000
Rejections at α=0.05               = 349
Power (lagged recipe, CANONICAL)   = 0.1745

--- Recipe B: CONTEMPORANEOUS tokens partialling (ALTERNATIVE / DIAGNOSTIC) ---
Residual SD (contemporaneous)      = 0.787015
MDES effect (0.40 × residual SD)   = 0.314806
HAC L (contemporaneous, T=28) = 3
true β under H1 (contemporaneous)  = 0.186304
Monte Carlo draws B                = 2000
Rejections at α=0.05               = 1423
Power (contemporaneous, diagnostic) = 0.7115

POWER RECIPE COMPARISON (v0.2.10 §0.10 Amendment #8 — Z3 REVERTED)
  Canonical   (LAGGED)        power = 0.1745  gate = FAIL
  Alternative (contemporaneous) power = 0.7115  gate = PASS
  Power threshold (demonstration-grade) = 0.5
Per v0.2.10 §0.10 Amendment #8 (REVERTS v0.2.9 §0.9):
  CANONICAL = LAGGED tokens partialling (matches regression k=1 lag structure).
  The CONTEMPORANEOUS recipe is preserved as documented diagnostic.
  The LAGGED recipe gates the verdict.


### Interpretation — Trio 4 (power, dual-recipe per v0.2.10 §0.10 Amendment #8)

Two power figures are reported above for the two-sided HAC test at MDES = 0.40 × residual-SD with $B = 2000$ Monte Carlo draws under the alternative DGP:

- **Canonical (LAGGED) recipe** — partials out $|\\Delta\\ln\\text{Tokens}|_{t-1}$ from $|\\Delta\\ln\\text{NotionalCost}^{USD}|_t$ on the $T_{\\text{lag}} = 27$ lag-aligned slice. **This matches R4-S3-USD's k=1 specification's actual lag structure** (the regression's own residuals from $|\\Delta\\ln \\text{NotionalCost}^{USD}|_t$ on $|\\Delta\\ln \\text{USDCOP}|_{t-1}, |\\Delta\\ln \\text{Tokens}|_{t-1}$). Per v0.2.10 §0.10 Amendment #8 (REVERTS v0.2.9 §0.9) this figure **gates the verdict**.
- **Alternative (contemporaneous) recipe** — partials out $|\\Delta\\ln\\text{Tokens}|_t$ from $|\\Delta\\ln\\text{NotionalCost}^{USD}|_t$ on the full $T = 28$ post-first-diff series; this is the Task 11 EDA Trio 5 recipe. v0.2.9 §0.9 had pinned this as canonical on chronological-precedence + first-principles grounds; **v0.2.10 §0.10 Amendment #8 withdraws that pin** as honest-disclosure-as-process (the "first-principles" defense overstated the case — the natural reading of CORRECTIONS-J's text "partialling out |Δln tokens|" in a spec where the regression uses |Δln tokens|_{t-1} is to use the SAME lagged regressor for the power-calc residual). Reported for transparency; **does NOT gate**.

The 0.50 floor is the pinned demonstration-grade threshold (CORRECTIONS-J + CORRECTIONS-U + §2.3.1 Amendment #5). Below this, a non-rejection cannot be interpreted as evidence for the null. **Anti-fishing safeguard (v0.2.10):** reverting to the lagged recipe is a TIGHTENING of the verdict's confidence claim, not a loosening — lagged power 0.17 < 0.50 fails; contemporaneous power 0.71 > 0.50 passes; v0.2.10 accepts the harder verdict. **No threshold is tuned.**

Anticipated outcome (per CORRECTIONS-U): canonical (lagged) power ≈ 0.17 < 0.50 → **POWER-HALT**.

## Verdict cell

In [6]:
# Gate evaluation per v0.2.10 §2.2.B + §0.10 Amendment #8 (Z3 REVERTED).
# Canonical power gate uses the LAGGED recipe (power_lagged); v0.2.9's
# contemporaneous pin is withdrawn.
# v0.2.10 §0.10 Amendment #8: PARTIAL-* labels are REMOVED for this Task 14
# verdict — they were premised on the power gate passing under the contemporaneous
# recipe. Under the reverted canonical recipe, the power gate FIRES FIRST.
gate_p = p_two_sided_primary < ALPHA_LEVEL
gate_n = T >= N_MIN
gate_power = power_lagged >= POWER_THRESHOLD

# v0.2.10 verdict routing — POWER-HALT fires before any p/N gates per Amendment #8.
# PARTIAL-* labels are NOT applicable to this verdict because the power gate
# fires first under the REVERTED canonical recipe.
if not gate_power:
    verdict = 'POWER-HALT'
elif gate_n and gate_p:
    verdict = 'REJECT_NULL'
elif gate_n and not gate_p:
    verdict = 'FAIL_TO_REJECT'
else:
    # Sub-floor N with passing power: this branch should not fire under v0.2.10
    # Amendment #8 because lagged power 0.17 < 0.50 makes gate_power = False.
    # Preserved defensively; if it ever fires, route to HALT for disposition memo.
    verdict = 'HALT'

assert verdict in VERDICT_LABELS, f'verdict {verdict!r} not in pinned label set'

print('=' * 64)
print('R4-S3-USD BEHAVIORAL TEST VERDICT (v0.2.10 §2.2.B + §0.10 Amendment #8)')
print('=' * 64)
print(f'k=1 primary:    α̂₁^USD = {alpha_1_usd_primary:+.6f}, p_2s = {p_two_sided_primary:.6f}')
print(f'k=5 sensit.:    α̂₁^USD = {alpha_1_usd_sens:+.6f}, p_2s = {p_two_sided_sens:.6f}')
print(f'N observed:     {T}     N_MIN spec floor: {N_MIN}')
print(f'Power CANONICAL (LAGGED, v0.2.10 Amendment #8): {power_lagged:.4f}  Threshold: {POWER_THRESHOLD}')
print(f'Power ALTERNATIVE (contemporaneous, diagnostic): {power_contemporaneous:.4f}  (does NOT gate)')
print('=' * 64)
print()
print('Gate evaluation (canonical LAGGED recipe per v0.2.10 §0.10 Amendment #8):')
print(f'  Power_lagged >= 0.50?          {gate_power}    (gates FIRST)')
print(f'  p < 0.05?                      {gate_p}      (not reached — power HALT fires)')
print(f'  N >= 38?                       {gate_n}      (not reached — power HALT fires)')
print()
print(f'VERDICT: {verdict}')
print()
if verdict == 'POWER-HALT':
    print('POWER-HALT routing per CORRECTIONS-U:')
    print('  - Disposition memo MUST be authored at:')
    print('    notebooks/dev_ai_cost_v2/dispositions/2026-05-18-task14-power-halt.md')
    print('  - Three pre-enumerated pivot options (A/B/C) per power_halt_template.md')
    print('  - User-selected pivot recorded; CORRECTIONS block (if any) added to spec')
    print('  - Post-hoc 3-way review before resuming')
    print()
    print('Anticipated per CORRECTIONS-U: Model QA\'s independent calc at T=38 gave')
    print('~0.54–0.66 with HAC small-sample inflation, so this HALT is a high-probability')
    print('operational event. Honest-disclosure-as-process per v0.2.10 §0.10 Amendment #8.')
print('=' * 64)

# v0.2.10 transparency footnote — what would the verdict be under the alternative recipe?
if power_contemporaneous >= POWER_THRESHOLD:
    print()
    print('Amendment #8 transparency: under the ALTERNATIVE (contemporaneous) recipe,')
    print(f'power_contemporaneous = {power_contemporaneous:.4f} >= {POWER_THRESHOLD}, which under v0.2.9')
    print('would have routed to PARTIAL-FAIL_TO_REJECT (N=28 < 38, p_2s>0.05).')
    print('The CANONICAL recipe per v0.2.10 §0.10 Amendment #8 is LAGGED (matches')
    print('the regression\'s own lag structure); that recipe gates the verdict above.')
    print('v0.2.10 accepts the HARDER verdict (POWER-HALT > PARTIAL-FAIL_TO_REJECT)')
    print('as a tightening of the confidence claim — no threshold is tuned.')


R4-S3-USD BEHAVIORAL TEST VERDICT (v0.2.10 §2.2.B + §0.10 Amendment #8)
k=1 primary:    α̂₁^USD = -18.116337, p_2s = 0.652410
k=5 sensit.:    α̂₁^USD = -35.750229, p_2s = 0.240405
N observed:     28     N_MIN spec floor: 38
Power CANONICAL (LAGGED, v0.2.10 Amendment #8): 0.1745  Threshold: 0.5
Power ALTERNATIVE (contemporaneous, diagnostic): 0.7115  (does NOT gate)

Gate evaluation (canonical LAGGED recipe per v0.2.10 §0.10 Amendment #8):
  Power_lagged >= 0.50?          False    (gates FIRST)
  p < 0.05?                      False      (not reached — power HALT fires)
  N >= 38?                       False      (not reached — power HALT fires)

VERDICT: POWER-HALT

POWER-HALT routing per CORRECTIONS-U:
  - Disposition memo MUST be authored at:
    notebooks/dev_ai_cost_v2/dispositions/2026-05-18-task14-power-halt.md
  - Three pre-enumerated pivot options (A/B/C) per power_halt_template.md
  - User-selected pivot recorded; CORRECTIONS block (if any) added to spec
  - Post-hoc 3-way rev

## Final disclaimers

**CORRECTIONS-S framing recap.** This notebook (Task 14, R4-S3-USD) is the *framework-relevant arm* of the R4-S3 family. R4-S3-COP (Task 13) was a *consistency check* — it confirmed mechanical FX pass-through but was not verdict-bearing under the (Y, M, X) framework; its CONSISTENCY-FAIL was reclassified to REGIME-CONDITIONAL per v0.2.8 §0.8 Z2 (both preconditions verified). **Per v0.2.10 §0.10 Amendment #6/#7, R4-S3-COP is UNINFORMATIVE for R5 corroboration and is REMOVED from any corroboration list.**

**Convergent evidence — v0.2.10 §0.10 Amendment #4 corrected accounting.** The "FOUR independent measurements" framing (R5, Z-arms, R4-S3-COP, R4-S3-USD) used in v0.2.8/v0.2.9 is **WITHDRAWN**. The corrected accounting is **ONE descriptive ratio (R5 PRIMARY, FX share ≈ 0.003%) + ONE behavioral test (R4-S3-USD)**. Z-arms (Notebook 06) are SAME-RATIO corroborations of R5 (NOT independent). R4-S3-COP (Notebook 03) is zero-power in this regime, removed per Amendment #7. Under v0.2.10 §0.10 Amendment #8, R4-S3-USD POWER-HALTs under the reverted canonical (lagged) recipe; the small-FX reading therefore rests on **ONE descriptive ratio + ONE POWER-HALTed behavioral test**, not "multiple convergent measurements". Honest disclosure per v0.2.10 §7 anti-fishing.

**π̂ disclaimer (CORRECTIONS-Y-6).** `ephemeral_pi_share = 0.398977` is a calibrated constant, not a time-series measurement. Sensitivity to this constant is bounded by Task 11/EDA tornado plots.

**Y-9 closure.** Cost data is at 1.000000× ccusage parity (UTC-aligned). LHS and RHS are apples-to-apples on the same UTC weekday cadence.

**Anti-fishing reminder.** Pre-pinned: LAG_PRIMARY=1, LAG_SENSITIVITY=5, HAC_LAGS=⌊T^(1/3)⌋, ALPHA_LEVEL=0.05 TWO-SIDED, MDES=0.40 residual-SD, power threshold 0.50 (demonstration-grade per §2.3.1), B=2000 MC draws. None of these were tuned after seeing the result. The v0.2.10 §0.10 Amendment #8 revert from contemporaneous to lagged canonical recipe is a TIGHTENING of the verdict's confidence claim (the harder verdict passes) — no threshold is tuned. The contemporaneous recipe is preserved as a documented diagnostic.

**v0.2.10 §0.10 Amendment #8 disclosure (Task 14).** Per v0.2.10 §0.10 Amendment #8, the canonical R4-S3-USD power recipe is **LAGGED tokens** partialling (matches the regression's k=1 spec lag structure). Under the canonical recipe: **power = 0.17 < 0.50** demonstration-grade floor → **POWER-HALT**. The CONTEMPORANEOUS recipe (power = 0.71) is preserved as a diagnostic. **Verdict: POWER-HALT per spec §2.2.B + §0.10 Amendment #8**, anticipated per CORRECTIONS-U; disposition memo required at `notebooks/dev_ai_cost_v2/dispositions/2026-05-18-task14-power-halt.md`. The v0.2.9 PARTIAL-FAIL_TO_REJECT label is **WITHDRAWN** (it was premised on the contemporaneous recipe passing the power gate; under the reverted canonical the power gate fires FIRST).

**v0.2.10 §0.10 Amendment #5 disclosure — demonstration-grade pin.** This iteration is pinned to **demonstration-grade** (N_MIN=38, POWER_MIN=0.50, MDES_SD=0.40 — below project defaults 75 / 0.80). The Task 17 verdict memo MUST state this disclosure. Population-scale claims are NOT permitted from this iteration's outputs.

## Disposition memo — POWER-HALT (v0.2.10 §0.10 Amendment #8 + CORRECTIONS-U routing)

Per CORRECTIONS-U pre-enumerated routing, a power-HALT disposition memo MUST be authored when this notebook's verdict cell fires POWER-HALT. The cell below writes the disposition memo at `notebooks/dev_ai_cost_v2/dispositions/2026-05-18-task14-power-halt.md` with the measured power figures and the three pre-enumerated pivot options (A: expand T; B: lower MDES; C: accept lower-power result). The user selects the pivot; CORRECTIONS block (if any) is added to spec; post-hoc 3-way review precedes resumption.

In [7]:
from pathlib import Path
import datetime as _dt

# Only write the disposition memo when POWER-HALT actually fires.
if verdict == 'POWER-HALT':
    memo_path = Path('dispositions/2026-05-18-task14-power-halt.md')
    memo_path.parent.mkdir(parents=True, exist_ok=True)
    memo_text = f"""# Disposition memo — Task 14 R4-S3-USD POWER-HALT (2026-05-18)

**Spec:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.10 §0.10
CORRECTIONS-W3 Amendment #8 (Z3 REVERTED to LAGGED canonical recipe).
**Plan:** Task 14, post-Wave-0 closure.
**Notebook:** `notebooks/dev_ai_cost_v2/04_r4s3_usd_behavioral.ipynb`.

## Trigger

Power-measurement HALT-checkpoint fired in Trio 4:
- Canonical recipe per v0.2.10 §0.10 Amendment #8 = **LAGGED tokens partialling**
  (matches R4-S3-USD k=1 spec's lag structure).
- Measured power (canonical) = **{{power_lagged:.4f}}**  (< 0.50 demonstration-grade floor).
- Alternative recipe (contemporaneous, diagnostic) = **{{power_contemporaneous:.4f}}**.
- N observed = {{T_obs}} (< spec floor N_MIN = {{N_MIN_val}}; not reached because power HALT fires first).
- Residual SD (canonical / lagged) = {{residual_sd_lagged:.6f}}.
- MDES effect at 0.40 × residual SD = {{mdes_effect_lagged:.6f}}.

**Anticipated per CORRECTIONS-U**: Model QA's independent calc at T=38 gave
~0.54–0.66 with HAC small-sample inflation, so this HALT is a high-probability
operational event. v0.2.10 §0.10 Amendment #8 reverts the v0.2.9 §0.9
contemporaneous pin to honest disclosure — the lagged recipe matches the
regression's own lag structure and is the natural reading of CORRECTIONS-J.

## Pre-enumerated user pivots (from `dispositions/power_halt_template.md`)

### Option A: Expand T by waiting
- Pause iteration; resume when `max(weekday days observed) ≥ 38` (spec N_MIN).
- Re-run from Task 10 (panel build) at the new window.
- Decision-citation block: <fill at user-selection time>

### Option B: Lower MDES to 0.30 residual-SD (documented downgrade)
- Update §2.2.B power threshold in a v0.2.11 patch.
- Disclose downgrade in §0 CORRECTIONS-X-NEXT.
- Decision-citation block: <fill at user-selection time>

### Option C: Accept lower-power result with explicit caveat in headline
- Proceed past HALT with documented canonical (lagged) power = {{power_lagged:.4f}}.
- Add caveat banner to `04_r4s3_usd_behavioral.ipynb` headline cell.
- Decision-citation block: <fill at user-selection time>

## HALT routing chain
Per `memory/feedback_pathological_halt_anti_fishing_checkpoint.md`:
1. HALT + this disposition memo (DONE)
2. User-enumerated pivot (A/B/C) — pending
3. CORRECTIONS block in spec — if user pivot adds a threshold change
4. Post-hoc 3-way review before resuming

## Status
- [x] HALT fired (date: 2026-05-18; canonical lagged power {{power_lagged:.4f}} < 0.50)
- [ ] User pivot selected: ____
- [ ] Disposition committed (this file is staged; commit follows user-pivot decision)

## v0.2.10 audit-trail context

This memo supersedes the v0.2.9 PARTIAL-FAIL_TO_REJECT verdict recorded in
commit `90f099e` (2026-05-17, notebook 04 Z3 power-recipe pin + PARTIAL-*
verdict under contemporaneous canonical). The v0.2.9 commit remains in git
history for audit trail; v0.2.10 §0.10 Amendment #8 reverts the canonical
recipe to lagged on honest-disclosure-as-process grounds (the v0.2.9
"first-principles" defense for contemporaneous overstated the case).

The corroboration accounting is now **ONE descriptive ratio (R5) + ONE
POWER-HALTed behavioral test (R4-S3-USD)** per v0.2.10 §0.10 Amendment #4
(Z-arms and R4-S3-COP are same-ratio corroborations / removed; not
independent measurements).
""".format(
        power_lagged=power_lagged,
        power_contemporaneous=power_contemporaneous,
        T_obs=T,
        N_MIN_val=N_MIN,
        residual_sd_lagged=residual_sd_lagged,
        mdes_effect_lagged=mdes_effect_lagged,
    )
    memo_path.write_text(memo_text)
    print(f'Disposition memo written: {memo_path}')
    print(f'  Canonical (lagged) power     = {power_lagged:.4f}')
    print(f'  Alternative (contemp) power  = {power_contemporaneous:.4f}')
    print(f'  N = {T} (< N_MIN = {N_MIN}, not reached)')
else:
    print(f'Verdict = {verdict}; no disposition memo required.')


Disposition memo written: dispositions/2026-05-18-task14-power-halt.md
  Canonical (lagged) power     = 0.1745
  Alternative (contemp) power  = 0.7115
  N = 28 (< N_MIN = 38, not reached)
